In [6]:
!pip install pm4py

In [7]:
import pandas as pd
import pm4py
import matplotlib.pyplot as plt

print("PM4Py version:", pm4py.__version__)
print("All libraries imported successfully!")

PM4Py version: 2.7.23.11
All libraries imported successfully!


In [8]:
df = pd.read_csv("HospitalOPD-noname.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (62422, 25)


,record_no,patient_id,Unnamed: 2,appointment_id,appointment_start_datetime,prescription_doctor_id,prescription_doctor_name,prescription_order_id,prescription_start_datetime,prescription_end_datetime,...,imaging_result_id,imaging_name,imaging_start_datetime,imaging_end_datetime,pharmacy_sale_line_id,medicine_name,pharmacy_start_datetime,pharmacy_end_datetime,flow_code,flow_name
0,1,10,NaN,445911,3/30/2026 5:36,225,Admin,106357,3/30/2026 5:38,3/30/2026 6:48,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Appointment → Prescription
1,2,57,NaN,335085,10/14/2025 5:23,193,Dr. Abrar Ahmed Bhatti,43515,10/14/2025 5:39,10/14/2025 5:53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Appointment → Prescription
2,3,57,NaN,335085,10/14/2025 5:23,193,Dr. Abrar Ahmed Bhatti,43541,10/14/2025 6:11,10/14/2025 6:24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Appointment → Prescription
3,4,60,NaN,322358,9/25/2025 13:36,195,Dr. Anique Ahmed,34654,9/25/2025 13:50,9/25/2025 13:51,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Appointment → Prescription
4,5,72,NaN,337714,10/17/2025 6:40,202,Dr. Mehreen Batool Bokhari,45263,10/17/2025 6:49,10/17/2025 6:49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,Appointment → Prescription


In [9]:
import pandas as pd

df = df.dropna(axis=1, how="all").copy()

activity_columns = {
    "Appointment": "appointment_start_datetime",
    "Prescription": "prescription_start_datetime",
    "Lab": "lab_start_datetime",
    "Imaging": "imaging_start_datetime",
    "Pharmacy": "pharmacy_start_datetime"
}

event_logs = []

for activity, timestamp_column in activity_columns.items():

    temp = df[["appointment_id", timestamp_column]].copy()

    temp[timestamp_column] = pd.to_datetime(
        temp[timestamp_column],
        errors="coerce"
    )

    temp = temp.dropna(
        subset=["appointment_id", timestamp_column]
    )

    temp = temp.rename(columns={
        "appointment_id": "Case_ID",
        timestamp_column: "Timestamp"
    })

    temp["Case_ID"] = temp["Case_ID"].astype(str)
    temp["Activity_Name"] = activity

    event_logs.append(
        temp[["Case_ID", "Activity_Name", "Timestamp"]]
    )

event_log_df = pd.concat(
    event_logs,
    ignore_index=True
)

event_log_df = event_log_df.sort_values(
    ["Case_ID", "Timestamp", "Activity_Name"]
).reset_index(drop=True)

event_log_df.to_csv(
    "careflow_normalized_event_log.csv",
    index=False
)

print("Normalized event log created successfully!")
print("Total events:", len(event_log_df))
print("Total cases:", event_log_df["Case_ID"].nunique())

display(event_log_df.head(10))

Normalized event log created successfully!
Total events: 148686
Total cases: 49912


,Case_ID,Activity_Name,Timestamp
0,112508,Appointment,2024-10-28 05:46:00
1,112508,Prescription,2025-10-03 06:46:00
2,124174,Appointment,2024-11-14 05:46:00
3,124174,Prescription,2024-11-26 06:44:00
4,139157,Appointment,2024-12-09 14:51:00
5,139157,Prescription,2025-08-25 06:03:00
6,142459,Appointment,2024-12-16 07:41:00
7,142459,Prescription,2024-12-17 04:48:00
8,142985,Appointment,2024-12-17 03:59:00
9,142985,Prescription,2024-12-17 04:24:00


In [10]:
print("Dataset shape:", event_log_df.shape)

print("\nMissing values:")
print(event_log_df.isnull().sum())

print("\nActivity counts:")
print(event_log_df["Activity_Name"].value_counts())

print("\nUnique cases:")
print(event_log_df["Case_ID"].nunique())

print("\nDuplicate events:")
print(event_log_df.duplicated().sum())

print("\nTimestamp range:")
print("Start:", event_log_df["Timestamp"].min())
print("End:", event_log_df["Timestamp"].max())

Dataset shape: (148686, 3)

Missing values:
Case_ID          0
Activity_Name    0
Timestamp        0
dtype: int64

Activity counts:
Activity_Name
Appointment     62422
Prescription    62410
Lab             18029
Imaging          5523
Pharmacy          302
Name: count, dtype: int64

Unique cases:
49912

Duplicate events:
36169

Timestamp range:
Start: 2024-04-25 11:12:00
End: 2026-07-23 16:43:00


In [11]:
duplicate_events = event_log_df[
    event_log_df.duplicated(
        subset=["Case_ID", "Activity_Name", "Timestamp"],
        keep=False
    )
].sort_values(
    ["Case_ID", "Timestamp", "Activity_Name"]
)

print("Duplicate event rows:", len(duplicate_events))

display(duplicate_events.head(20))

Duplicate event rows: 49655


,Case_ID,Activity_Name,Timestamp
22,200905,Appointment,2025-03-20 03:55:00
23,200905,Appointment,2025-03-20 03:55:00
24,200905,Appointment,2025-03-20 03:55:00
44,203343,Appointment,2025-03-25 05:42:00
45,203343,Appointment,2025-03-25 05:42:00
46,203343,Appointment,2025-03-25 05:42:00
47,203343,Appointment,2025-03-25 05:42:00
48,203343,Prescription,2025-03-25 05:53:00
49,203343,Prescription,2025-03-25 05:53:00
50,203343,Prescription,2025-03-25 05:53:00


In [12]:
pm4py_log = pm4py.format_dataframe(
    event_log_df,
    case_id="Case_ID",
    activity_key="Activity_Name",
    timestamp_key="Timestamp"
)

print("PM4Py event log prepared successfully!")

PM4Py event log prepared successfully!


In [13]:
dfg, start_activities, end_activities = pm4py.discover_dfg(pm4py_log)

print("Process discovery completed!")
print("\nStart activities:", start_activities)
print("\nEnd activities:", end_activities)

print("\nActivity transitions:")
for (source, target), frequency in sorted(
    dfg.items(), key=lambda item: item[1], reverse=True
):
    print(f"{source} -> {target}: {frequency}")

Process discovery completed!

Start activities: {'Appointment': 49494, 'Lab': 131, 'Imaging': 282, 'Prescription': 5}

End activities: {'Prescription': 41611, 'Lab': 5418, 'Imaging': 2773, 'Pharmacy': 95, 'Appointment': 15}

Activity transitions:
Appointment -> Prescription: 47939
Appointment -> Appointment: 12510
Prescription -> Prescription: 12417
Lab -> Lab: 11618
Prescription -> Lab: 5296
Prescription -> Imaging: 2975
Imaging -> Prescription: 1664
Appointment -> Imaging: 1600
Imaging -> Lab: 614
Lab -> Imaging: 481
Lab -> Prescription: 385
Appointment -> Lab: 358
Imaging -> Appointment: 288
Pharmacy -> Pharmacy: 194
Imaging -> Imaging: 184
Lab -> Appointment: 125
Prescription -> Pharmacy: 106
Pharmacy -> Lab: 12
Prescription -> Appointment: 5
Lab -> Pharmacy: 2
Pharmacy -> Imaging: 1


In [14]:
pm4py.save_vis_dfg(
    dfg,
    start_activities,
    end_activities,
    "careflow_pm4py_process_map.png"
)

print("Process map saved successfully!")

Process map saved successfully!


In [1]:
import shutil
print("Graphviz dot location:", shutil.which("dot"))

Graphviz dot location: C:\Users\nehas\Downloads\Graphviz\bin\dot.EXE


In [15]:
import pandas as pd

summary_rows = [
    {
        "Source_Activity": source,
        "Target_Activity": target,
        "Frequency": frequency
    }
    for (source, target), frequency in dfg.items()
]

summary_df = pd.DataFrame(summary_rows)

summary_df = summary_df.sort_values(
    by="Frequency",
    ascending=False
)

summary_df.to_csv(
    "careflow_process_discovery_summary.csv",
    index=False
)

print("Process discovery summary saved successfully!")
display(summary_df.head(10))

Process discovery summary saved successfully!


,Source_Activity,Target_Activity,Frequency
3,Appointment,Prescription,47939
0,Appointment,Appointment,12510
20,Prescription,Prescription,12417
10,Lab,Lab,11618
18,Prescription,Lab,5296
17,Prescription,Imaging,2975
7,Imaging,Prescription,1664
1,Appointment,Imaging,1600
6,Imaging,Lab,614
9,Lab,Imaging,481
